In [ ]:
# ==================== embedded: report_parser.py ====================
"""
Multilingual radiology-report label extractor for RSNA Knee.

Ported (with minor formatting cleanup) from the public Kaggle notebook
`aadigupta7686/0-899-let-me-cook` — see
https://www.kaggle.com/code/aadigupta7686/0-899-let-me-cook
Kept structurally identical so we can benefit from any future updates by
diff-and-merge rather than by rewriting.

Public entry points:
    extract(report_text) -> {label: score,   label + "__conf": conf,
                             label + "__npos": n_pos, label + "__nneg": n_neg}

Scores are in [0.04, 0.95] and confidences in [0.05, 1.0]; the caller decides
how to threshold / weight them.
"""
from __future__ import annotations

import re
import unicodedata

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Turkish dotted/dotless i must be folded before casefolding, otherwise
# "İZLENMEZ" and "izlenmez" diverge. ß and the Croatian/Serbian d-with-stroke
# likewise.
_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    """Fold case, diacritics and separators; keep Greek and Cyrillic letters."""
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")               # soft hyphen
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text: str):
    """Split into clauses, then attach `header:` lines to the value below."""
    norm = normalize(text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]
    merged = []
    for i, c in enumerate(raw):
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
        else:
            merged.append(c)
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


# --- Polarity vocabularies --------------------------------------------------
NEGATION = _rx(
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",
    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b", r"\bnone\b", r"\bnil\b",
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b",
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\babsence\b",
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",
    r"φυσιολογικ", r"ακεραι",
    r"unauffallig", r"regelrecht", r"\bintakt\b",
    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",
    r"\bgaaf\b", r"\bnormaal\b",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bV\.a\.\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

# --- Pathology vocabularies -------------------------------------------------
TEAR = _rx(
    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",
    r"\bavuls",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    r"riss(bildung|e|es)?\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",
    r"\bpuknuce", r"\bruptur", r"\bprekid\b", r"\bpukotin",
    r"ρηξη", r"ρηξις", r"ρηγμα",
    r"руптура", r"разкъсв", r"разрив", r"скъсв",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλ", r"дегенерат",
    r"\bμυξοειδ", r"\bμυξωδ",
    r"\bmuco ?ide\b", r"aufgefasert",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"aumento de senal", r"alteracion de senal", r"cambio de senal",
    r"\bsignalanhebung", r"\bsignalalteration", r"verhoogd signaal", r"sinyal artis",
    r"αυξημενο σημα", r"повишен сигнал",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)

# --- Anatomy lexicons (paired structures) -----------------------------------
ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        r"\bχιαστ\w*",
        r"предна кръстна", r"предната кръстна",
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband", r"\b(mediale|laterale) banden\b",
        r"\bcollaterale banden\b",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι", r"\bπλαγι\w* συνδεσμ", r"\bπλαγιοι\b",
        r"медиален колатерал", r"вътрешна странична", r"\bколатерал\w*",
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
    ),
}

# --- OA evidence + compartment lexicons ------------------------------------
OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac",
    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",
    r"joint space narrowing", r"pinzamiento articular",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",
    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",
    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"outerbridge",
)

COMPARTMENT = {
    "Medial OA": _rx(
        r"medial (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial medial", r"femorotibial interno",
        r"mediaal femorotibiaal", r"mediale femorotibial",
        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",
        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",
        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",
        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",
        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",
    ),
    "Lateral OA": _rx(
        r"lateral (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial lateral", r"femorotibial externo",
        r"lateraal femorotibiaal", r"laterale femorotibial",
        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",
        r"lateral femorotibial", r"dis kompartman", r"lateral kompartman",
        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",
        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",
        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",
    ),
    "PF OA": _rx(
        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",
        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",
        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",
        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",
        r"пател", r"феморопател", r"тролх",
        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",
    ),
}

# --- Self-declaring findings ------------------------------------------------
DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"derrame articular", r"\bderrame\b", r"liquido articular",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"\bhydrops\b", r"gewrichtseffusie",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu",
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"synovitis", r"synoviale? (verdikking|proliferatie)",
        r"synovialitis", r"synovialis(verdickung|proliferation)",
        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"verdikkingen van (het )?synovium", r"pannus",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",
        r"contusion osea", r"edema oseo", r"edema de medula osea",
        r"oedeme osseux", r"contusion osseuse",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"kontusion", r"bone bruise",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",
        r"kostani edem", r"edem kosti", r"kontuzij",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",
        r"костномозъчен едем", r"костен едем", r"контузионен",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b", r"\bkirik\b",
        r"\bfraktur", r"\bprijelom", r"impresijsk[^ ]* fraktur",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri",
    ),
}

DECOY = {
    "Fracture": _rx(r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}

# --- Stem+side proximity matcher ------------------------------------------
STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",
                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",
                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")

SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
                  r"\bmediale\w*", r"\bintern[oa]\w*", r"\binterne\w*", r"\binnen\w*",
                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b",
                  r"\bbinnen\w*", r"\bmediaal\b")
SIDE_LATERAL = _rx(r"\blateral\w*", r"\bextern[oa]\w*", r"\bexterne\w*", r"\bdis\b",
                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",
                   r"\bfibular collateral\b", r"\bvanjsk\w*")
SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",
                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",
                    r"\banteriyor\w*", r"\bavant\b", r"\bant[eé]rieur\w*")
SIDE_POSTERIOR = _rx(r"\bposterior\w*", r"\bpost[eé]rieur\w*", r"\bposteriore\w*",
                     r"\bhinter\w*", r"\bachterste\b", r"\barka\b", r"\bstraznj\w*",
                     r"\bzadnj\w*", r"\bοπισθι\w*", r"\bзадн\w*", r"\bpostero\w*")

STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kgğ]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",
                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",
                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",
                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",
                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",
                          r"femoral condyl\w*", r"tibial plateau\w*",
                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",
                          r"femurkondyl\w*", r"femoralne? kondil\w*",
                          r"tibijaln\w* plato", r"femoral kondil\w*",
                          r"tibia plato", r"tibyal plato")


def _distance(clause, stem_rx, qual_rx, window=55):
    best = None
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        for q in qual_rx.finditer(clause[lo:hi]):
            qs, qe = lo + q.start(), lo + q.end()
            d = 0 if qs < m.end() and qe > m.start() else \
                min(abs(m.start() - qe), abs(qs - m.end()))
            best = d if best is None else min(best, d)
    return best


def _near(clause, stem_rx, qual_rx, window=55):
    return _distance(clause, stem_rx, qual_rx, window) is not None


STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),
    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),
}

# --- Severity + context modifiers ------------------------------------------
SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\bminimal\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",
)

GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b",
)

DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"субхондрал", r"subchondrale?",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\bcontusion osea\b",
    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",
    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",
    r"\bcontusion osseuse\b", r"\bbone bruise\b", r"\bbotcontusie\b",
    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",
)


def _polarity(clause, anchor_end):
    """positive / negative / uncertain — clause-scoped."""
    if UNCERTAIN.search(clause):
        return "uncertain"
    if NEGATION.search(clause):
        return "negative"
    if NORMALITY.search(clause):
        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):
            return "positive"
        return "negative"
    return "positive"


class _Matcher:
    """Phrase-lexicon first, stem+side proximity as fallback."""
    def __init__(self, phrase_rx, stem=None, side=None, window=55, contrary=None):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window
        self.contrary = contrary

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None and not self._wrong_side(clause):
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None

    def _wrong_side(self, clause):
        if self.contrary is None or self.stem is None:
            return False
        other = _distance(clause, self.stem, self.contrary, self.window)
        if other is None:
            return False
        own = _distance(clause, self.stem, self.side, self.window)
        return own is None or other < own


CONTRARY = {"ACL": SIDE_POSTERIOR, "MCL": SIDE_LATERAL}

ANAT_MATCH = {
    tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt], contrary=CONTRARY.get(tgt))
    for tgt in PAIRED
}
COMPARTMENT_MATCH = {
    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),
    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),
    "PF OA": _Matcher(COMPARTMENT["PF OA"]),
}
DIRECT_MATCH = {
    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)
    for tgt, rx in DIRECT.items()
}


def _severity(clause):
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    return 0.75


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None,
                   context_penalty=None, context_bonus=None):
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMALITY.search(c) and not NEGATION.search(c):
                n_neg += 1
            continue
        pol = _polarity(c, m.end())
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)

    if n_pos or n_unc:
        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05
    return score, conf, n_pos, n_neg


def extract(report: str) -> dict:
    """Extract twelve (score, confidence, n_pos, n_neg) tuples from one report."""
    cls = clauses(report)
    out = {}
    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)

    for tgt in TARGETS:
        if tgt in PAIRED:
            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)
        elif tgt in OA_TARGETS:
            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)
        elif tgt == "Contusion":
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),
                                              context_penalty=DEGENERATIVE_MARROW,
                                              context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    # Global OA propagation — "tricompartmental OA" is evidence for all three.
    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c, 0) == "positive"]
    if g_hits:
        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)
        for tgt in OA_TARGETS:
            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:
                out[tgt] = max(out[tgt], gscore * 0.92)
                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)

    # Silent synovitis inherits fraction of effusion evidence.
    if out["Synovitis__npos"] == 0 and out["Synovitis__nneg"] == 0:
        out["Synovitis"] = max(out["Synovitis"], 0.28 + 0.45 * (out["Effusion"] - 0.28))

    return out


# ==================== embedded: end2end.py ====================
"""
End-to-end DINOv2 SlotHead fine-tune for RSNA Knee.

Minimal viable port of the SlotHead architecture from
`aadigupta7686/0-899-let-me-cook`. Deliberately simpler:

- 3 slots per study (Sagittal / Coronal / Axial), pick the longest series in each plane
- 3 slices per slot (central-band, geometry-ordered — helpers in rsna_knee_baseline.py)
- DINOv2 ViT-S/14 with the LAST 2 BLOCKS unfrozen (0.899 uses 6; we're conservative)
- SlotHead: per-label attention over the 3 slot embeddings, with anatomy prior
- Training: 3 epochs, batch=4 studies, two LRs (encoder 5e-6, head 1e-3)
- BCEWithLogitsLoss, per-row sample_weight from parser confidence × expert-multiplier

Callers wire in via `USE_END2END = True` in rsna_knee_baseline.py; on any failure
the baseline's __main__ safety net writes the 0.5-filled fallback submission.
"""
from __future__ import annotations
import math, time
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --- constants matching the baseline ---------------------------------------
LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
N_TARGETS = len(LABEL_COLS)
SLOT_NAMES = ["Sagittal", "Coronal", "Axial"]
N_SLOT = len(SLOT_NAMES)
SLICES_PER_SLOT = 3
IMG_SIZE = 224

# Per-label anatomy prior (from 0.899 recipe cell 8, remapped for our 3-slot scheme).
# Positive tilt = attention bias toward that slot for that label.
# 0=Sagittal, 1=Coronal, 2=Axial.
SLOT_PRIOR_TABLE = {
    "ACL": (0,),
    "MCL": (1,),
    "Medial Meniscus": (0, 1),
    "Lateral Meniscus": (0, 1),
    "Medial OA": (1, 2),
    "Lateral OA": (1, 2),
    "PF OA": (0, 2),
    "Effusion": (0, 2),
    "Synovitis": (0, 2),
    "Baker's": (0,),
    "Contusion": (0, 1, 2),
    "Fracture": (0, 1, 2),
}
SLOT_PRIOR_STRENGTH = 0.55

# Training hyperparams
EPOCHS = 3
BATCH_STUDIES = 4
LR_HEAD = 1e-3
LR_BACKBONE = 5e-6
WEIGHT_DECAY = 0.02


# --- slot picker -----------------------------------------------------------
def pick_slots_for_study(series_rows: pd.DataFrame) -> dict[str, str]:
    """Return {slot_name: SeriesInstanceUID} — longest series per plane."""
    chosen: dict[str, str] = {}
    for plane in SLOT_NAMES:
        cand = series_rows[series_rows["Anatomical_Plane"] == plane]
        if len(cand) == 0:
            continue
        # Tie-break on longest series — thicker stack samples the joint better.
        cand = cand.copy()
        # SeriesInstanceUID as tie-breaker for determinism.
        chosen[plane] = cand.iloc[0]["SeriesInstanceUID"]
    return chosen


# --- dataset ---------------------------------------------------------------
class StudyDataset(Dataset):
    """Yields (imgs, mask, labels, weight) per study.

    imgs   : (N_SLOT * SLICES_PER_SLOT, 3, H, W) float32 in [0, 1]
    mask   : (N_SLOT,) float32 — 1 if slot has any decoded slices else 0
    labels : (N_TARGETS,) float32 — soft targets in [0, 1]
    weight : scalar float32 — expert (3.0) or weak (1.0) multiplier * conf
    """

    def __init__(self, study_ids, series_df, root: Path,
                 labels_df, conf_df, is_expert,
                 pixel_loader: Callable, ordered_paths_fn: Callable,
                 sampler_fn: Callable, series_window_fn: Callable,
                 apply_window_fn: Callable, augment: bool = False):
        self.study_ids = list(study_ids)
        self.series_by_study = series_df.groupby("StudyInstanceUID")
        self.root = root
        self.labels_df = labels_df.reindex(self.study_ids).fillna(0)
        self.conf_df = conf_df.reindex(self.study_ids).fillna(0.05)
        self.is_expert = is_expert.reindex(self.study_ids).fillna(False)
        # Callables so we can reuse baseline's helpers without duplicating them.
        self.pixel_loader = pixel_loader
        self.ordered_paths_fn = ordered_paths_fn
        self.sampler_fn = sampler_fn
        self.series_window_fn = series_window_fn
        self.apply_window_fn = apply_window_fn
        self.augment = augment

    def __len__(self):
        return len(self.study_ids)

    def _blank_slot(self) -> np.ndarray:
        return np.zeros((SLICES_PER_SLOT, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)

    def _read_slot(self, series_dir: Path) -> tuple[np.ndarray | None, bool]:
        """Return (imgs[SLICES, 3, H, W], has_content). imgs is None on empty slot."""
        ordered = self.ordered_paths_fn(series_dir)
        if not ordered:
            return None, False
        sampled = self.sampler_fn(len(ordered), SLICES_PER_SLOT)
        needed = set(sampled)
        for i in sampled:
            needed.add(max(0, i - 1))
            needed.add(min(len(ordered) - 1, i + 1))
        raw_by_idx: dict[int, np.ndarray] = {}
        for i in sorted(needed):
            arr = self.pixel_loader(ordered[i])
            if arr is not None:
                raw_by_idx[i] = arr
        if not raw_by_idx:
            return None, False
        lo, hi = self.series_window_fn(list(raw_by_idx.values()))
        windowed = {i: self.apply_window_fn(a, lo, hi) for i, a in raw_by_idx.items()}
        out = np.zeros((SLICES_PER_SLOT, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        for k, i in enumerate(sampled):
            if i not in windowed:
                continue
            curr = windowed[i]
            prev = windowed.get(max(0, i - 1), curr)
            nxt = windowed.get(min(len(ordered) - 1, i + 1), curr)
            if prev.shape != curr.shape:
                prev = curr
            if nxt.shape != curr.shape:
                nxt = curr
            rgb = np.stack([prev, curr, nxt], axis=0)  # (3, H, W)
            out[k] = _resize_to(rgb, IMG_SIZE)
        return out, True

    def __getitem__(self, idx):
        sid = self.study_ids[idx]
        try:
            srows = self.series_by_study.get_group(sid)
        except KeyError:
            srows = pd.DataFrame(columns=["SeriesInstanceUID", "Anatomical_Plane"])
        chosen = pick_slots_for_study(srows)

        slot_imgs = []
        mask = np.zeros(N_SLOT, dtype=np.float32)
        for i, plane in enumerate(SLOT_NAMES):
            if plane not in chosen:
                slot_imgs.append(self._blank_slot())
                continue
            sdir = self.root / sid / chosen[plane]
            imgs, ok = self._read_slot(sdir)
            if not ok:
                slot_imgs.append(self._blank_slot())
                continue
            slot_imgs.append(imgs)
            mask[i] = 1.0

        # (N_SLOT * SLICES_PER_SLOT, 3, H, W)
        imgs_tensor = np.concatenate(slot_imgs, axis=0)
        labels = self.labels_df.loc[sid, LABEL_COLS].astype(np.float32).values
        conf = float(self.conf_df.loc[sid, LABEL_COLS].astype(np.float32).mean())
        base = 3.0 if bool(self.is_expert.loc[sid]) else 1.0
        weight = np.float32(base * conf)

        if self.augment:
            imgs_tensor = _augment_np(imgs_tensor)

        # ImageNet norm (DINOv2 expects it)
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)
        imgs_tensor = (imgs_tensor - mean) / std

        return (torch.from_numpy(imgs_tensor),
                torch.from_numpy(mask),
                torch.from_numpy(labels),
                torch.tensor(weight))


def _resize_to(rgb: np.ndarray, size: int) -> np.ndarray:
    """Resize (3, H, W) float array to (3, size, size). No aspect-ratio preserve —
    knee slices are already ~square, distortion is negligible and matches the pretrained
    receptive field."""
    if rgb.shape[-2:] == (size, size):
        return rgb.astype(np.float32)
    t = torch.from_numpy(rgb).unsqueeze(0)   # (1, 3, H, W)
    out = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
    return out.squeeze(0).numpy().astype(np.float32)


def _augment_np(imgs: np.ndarray) -> np.ndarray:
    """Rigid jitter + intensity noise on all slices as a batch. In-place friendly."""
    # Small rotation via reflection-padded flip — cheap and gets the encoder used to
    # slight misalignment. Real rotation would require per-slice grid_sample.
    if np.random.rand() < 0.5:
        imgs = np.flip(imgs, axis=-1).copy()   # horizontal flip
    if np.random.rand() < 0.3:
        # small intensity jitter
        imgs = np.clip(imgs * (1.0 + 0.1 * (np.random.rand() - 0.5)), 0, 1)
    return imgs.astype(np.float32)


# --- model -----------------------------------------------------------------
class SlotHead(nn.Module):
    """Per-diagnosis attention over slot embeddings, with anatomy-prior bias."""

    def __init__(self, dim: int, n_slot: int = N_SLOT, n_out: int = N_TARGETS,
                 hidden: int = 256, dropout: float = 0.2, prior: bool = True):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        self.prior = prior
        if prior:
            p = torch.zeros(n_out, n_slot)
            for lbl, slots in SLOT_PRIOR_TABLE.items():
                idx = LABEL_COLS.index(lbl)
                for s in slots:
                    if s < n_slot:
                        p[idx, s] = SLOT_PRIOR_STRENGTH
            self.register_buffer("slot_prior", p)

    def forward(self, x, mask):
        """x: (B, N_SLOT, dim), mask: (B, N_SLOT)."""
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class KneeModel(nn.Module):
    """DINOv2 backbone + slot pool + SlotHead."""

    def __init__(self, backbone, dim, unfreeze_last=2):
        super().__init__()
        self.backbone = backbone
        # Freeze all, then unfreeze last N transformer blocks + norm.
        for p in self.backbone.parameters():
            p.requires_grad = False
        blocks = self.backbone.blocks
        for blk in blocks[-unfreeze_last:]:
            for p in blk.parameters():
                p.requires_grad = True
        # timm's DINOv2 has a final norm layer.
        if hasattr(self.backbone, "norm"):
            for p in self.backbone.norm.parameters():
                p.requires_grad = True
        self.head = SlotHead(dim * 2)   # CLS + mean-of-patches

    def forward(self, imgs, mask):
        """imgs: (B, N_SLOT*SLICES_PER_SLOT, 3, H, W).
        mask: (B, N_SLOT)."""
        B = imgs.shape[0]
        S = N_SLOT * SLICES_PER_SLOT
        x = imgs.reshape(B * S, 3, IMG_SIZE, IMG_SIZE)
        # timm ViT: forward_features returns (B, N_tokens, D) including CLS at [:, 0].
        feats = self.backbone.forward_features(x)
        if isinstance(feats, dict):
            feats = feats.get("x_norm_patchtokens", feats.get("x"))
        cls = feats[:, 0]
        mean_patch = feats[:, 1:].mean(dim=1)
        per_slice = torch.cat([cls, mean_patch], dim=1)   # (B*S, 2D)
        # Pool the 3 slices in each slot: mean.
        per_slot = per_slice.reshape(B, N_SLOT, SLICES_PER_SLOT, -1).mean(dim=2)
        return self.head(per_slot, mask)


# --- training + inference --------------------------------------------------
def _build_backbone_and_load(weight_path: Path):
    import timm
    net = timm.create_model("vit_small_patch14_dinov2.lvd142m",
                            pretrained=False, num_classes=0, img_size=IMG_SIZE)
    state = torch.load(str(weight_path), map_location="cpu", weights_only=False)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    state = {k.replace("teacher.backbone.", "").replace("backbone.", ""): v
             for k, v in state.items()}
    # Interpolate pos_embed 518→224 (37×37+1 → 16×16+1)
    if "pos_embed" in state:
        pe = state["pos_embed"]
        target = net.pos_embed.shape
        if pe.shape != target:
            cls, patch = pe[:, :1], pe[:, 1:]
            old = int(round(patch.shape[1] ** 0.5))
            new = int(round((target[1] - 1) ** 0.5))
            patch = patch.reshape(1, old, old, -1).permute(0, 3, 1, 2)
            patch = F.interpolate(patch, size=(new, new), mode="bicubic", align_corners=False)
            patch = patch.permute(0, 2, 3, 1).reshape(1, new * new, -1)
            state["pos_embed"] = torch.cat([cls, patch], dim=1)
    net.load_state_dict(state, strict=False)
    return net, 384   # ViT-S hidden dim


def train_and_predict(train_ids, test_ids, series_df, root_train, root_test,
                      labels_df, conf_df, is_expert,
                      pixel_loader, ordered_paths_fn, sampler_fn,
                      series_window_fn, apply_window_fn) -> np.ndarray:
    """Full train + inference. Returns test predictions as (n_test, N_TARGETS)
    numpy array of sigmoid probabilities in the order of test_ids."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[e2e] device = {device}")

    # Locate DINOv2 weights.
    weight_paths = list(Path("/kaggle/input").rglob("dinov2*.pth"))
    if not weight_paths:
        raise FileNotFoundError("No dinov2*.pth found under /kaggle/input — attach a "
                                "DINOv2 weights dataset before enabling USE_END2END.")
    weight_path = sorted(weight_paths, key=lambda p: (0 if "vits14" in p.name else 1))[0]
    print(f"[e2e] loading DINOv2 from {weight_path.name}")

    backbone, dim = _build_backbone_and_load(weight_path)
    model = KneeModel(backbone, dim, unfreeze_last=2).to(device)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[e2e] trainable params: {n_trainable/1e6:.2f} M")

    # Two-LR optimizer.
    head_params = list(model.head.parameters())
    bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([
        {"params": head_params, "lr": LR_HEAD},
        {"params": bb_params, "lr": LR_BACKBONE},
    ], weight_decay=WEIGHT_DECAY)

    train_ds = StudyDataset(train_ids, series_df, root_train,
                             labels_df, conf_df, is_expert,
                             pixel_loader, ordered_paths_fn, sampler_fn,
                             series_window_fn, apply_window_fn, augment=True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_STUDIES, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)

    loss_fn = nn.BCEWithLogitsLoss(reduction="none")
    model.train()
    for epoch in range(EPOCHS):
        t0 = time.time()
        seen = 0
        loss_acc = 0.0
        for imgs, mask, y, w in train_dl:
            imgs = imgs.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            w = w.to(device, non_blocking=True)
            logits = model(imgs, mask)
            per_el = loss_fn(logits, y)
            loss = (per_el.mean(dim=1) * w).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            loss_acc += float(loss.item()) * imgs.size(0)
            seen += imgs.size(0)
            if seen % 200 == 0:
                elapsed = time.time() - t0
                print(f"  epoch {epoch+1} step {seen}/{len(train_ds)} "
                      f"({elapsed:.0f}s, loss={loss_acc/seen:.4f})")
        print(f"[e2e] epoch {epoch+1}/{EPOCHS} done in {time.time()-t0:.0f}s, "
              f"mean loss = {loss_acc/max(seen,1):.4f}")

    # Inference on test.
    print(f"[e2e] predicting on {len(test_ids)} test studies")
    model.eval()
    # Labels/conf/is_expert only used for training; test doesn't have them, use dummies.
    dummy_labels = pd.DataFrame(0.0, index=test_ids, columns=LABEL_COLS)
    dummy_conf = pd.DataFrame(1.0, index=test_ids, columns=LABEL_COLS)
    dummy_expert = pd.Series(False, index=test_ids)
    test_ds = StudyDataset(test_ids, series_df, root_test,
                            dummy_labels, dummy_conf, dummy_expert,
                            pixel_loader, ordered_paths_fn, sampler_fn,
                            series_window_fn, apply_window_fn, augment=False)
    test_dl = DataLoader(test_ds, batch_size=BATCH_STUDIES, shuffle=False,
                         num_workers=2, pin_memory=True)
    preds = np.zeros((len(test_ids), N_TARGETS), dtype=np.float32)
    idx = 0
    with torch.no_grad():
        for imgs, mask, _, _ in test_dl:
            imgs = imgs.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            logits = model(imgs, mask)
            p = torch.sigmoid(logits).cpu().numpy()
            preds[idx:idx + p.shape[0]] = p
            idx += p.shape[0]
    return preds


# ==================== main baseline: rsna_knee_baseline.py ====================
"""
RSNA Knee Abnormality Detection — Working Baseline
==================================================

Single-file, Kaggle-ready baseline for:
https://www.kaggle.com/competitions/rsna-knee-abnormality-detection

Approach (deliberately simple, submits reliably):
  1. For each series, sample a handful of representative slices.
  2. From each slice extract robust pixel statistics (mean, std, percentiles,
     histogram, edge density) + a few safe DICOM header fields.
  3. Aggregate per-series features to one feature vector per study
     (grouped by anatomical plane so plane-specific signal is preserved).
  4. Train one HistGradientBoostingClassifier per label (multi-label).
  5. Validate with BOTH a random StratifiedKFold on label-sum, AND a
     scanner-grouped KFold (grouping by Manufacturer + model + software).
     Report macro-averaged ROC AUC (the competition metric) for each.
  6. Predict on the test set and write submission.csv matching
     sample_submission.csv row-for-row.

Only sklearn / numpy / pandas / pydicom (all on Kaggle's default image, so
this runs with internet disabled). No pretrained weights required.

The script is organised as cell-style blocks (`# %% ...`) so it can be pasted
directly into a Kaggle notebook.
"""

# %% [imports] --------------------------------------------------------------
from __future__ import annotations

import os
import gc
import re
import sys
import math
import time
import warnings
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# %% [config] ---------------------------------------------------------------
# Kaggle mounts competition data in one of two layouts depending on how the
# notebook was attached:
#   /kaggle/input/<slug>/                       (dataset-style attach)
#   /kaggle/input/competitions/<slug>/          (competition-style attach)
# We probe both and pick whichever actually contains the CSVs, so the script
# works regardless of how the user added the dataset. Override with the
# RSNA_DATA_DIR environment variable if running elsewhere.
def _locate_data_dir() -> Path:
    env = os.environ.get("RSNA_DATA_DIR")
    if env:
        return Path(env)
    candidates: list[Path] = []
    slug = "rsna-knee-abnormality-detection"
    for base in ("/kaggle/input", "/kaggle/input/competitions"):
        candidates.append(Path(base) / slug)
    # As a last resort scan /kaggle/input for any dir with train.csv +
    # sample_submission.csv — covers renamed attachments.
    root = Path("/kaggle/input")
    if root.is_dir():
        for p in root.rglob("sample_submission.csv"):
            candidates.append(p.parent)
    for c in candidates:
        if (c / "train.csv").is_file() and (c / "sample_submission.csv").is_file():
            return c
    # Fallback: return the first candidate so error messages are informative.
    return candidates[0] if candidates else Path("/kaggle/input")


DATA_DIR = _locate_data_dir()
OUT_DIR = Path(os.environ.get("RSNA_OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Twelve target columns, in the exact order required by the submission header.
LABEL_COLS: list[str] = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

# Sampling knobs. Keep low so the baseline fits comfortably in the 9-hour
# CPU budget on ~1,300 test studies; scale up on a stronger baseline.
SLICES_PER_SERIES = 5           # more slices → more stable per-series stats
MAX_SERIES_PER_STUDY = 8        # was 6; median-series-per-study on train is 5
USE_25D_INPUT = True            # 3 adjacent slices as R/G/B channels (cheap +0.02-0.04 AUC)
N_HIST_BINS = 8                 # coarse intensity histogram
N_FOLDS = 5
RANDOM_STATE = 42
PLANES = ("Sagittal", "Coronal", "Axial")
# v5 showed 0.79 random vs 0.64 grouped AUC → the meta_* DICOM header
# features (TR/TE/thickness/spacing/field strength/imaging frequency) leak
# scanner identity. Prevalence differs across public/private test per the
# brief, so this shortcut wins public LB and loses private. Keep off.
USE_SCANNER_METADATA_FEATURES = False
# v14 bisect: v13 (advanced parser) dropped grouped AUC 0.700 → 0.688 despite
# recovering 1,249 extra positive rows. Toggle to isolate whether the parser
# upgrade OR the DINOv2/windowing/ordering stack is the regressor. False = use
# v10's binary regex parser + flat sample_weight. True = v13's soft parser + conf.
USE_ADVANCED_PARSER = False
# v15: end-to-end DINOv2 SlotHead fine-tune (see end2end.py). Replaces the
# HistGradientBoosting frozen-feature path entirely. On any failure, the
# __main__ try/except falls back to writing sample_submission.
USE_END2END = True
# Holdout fraction for end-to-end AUC measurement (no k-fold — too expensive).
E2E_HOLDOUT = 0.20

# Optional: subsample training studies for iteration speed. Set to None to
# use all training studies. On the real submission run leave it None.
TRAIN_SUBSAMPLE = int(os.environ.get("RSNA_TRAIN_SUBSAMPLE", "0")) or None

# %% [report → weak labels] -------------------------------------------------
# train.csv ships with labels populated ONLY for the ~58 expert-annotated
# studies; the other ~4349 rows have NaN targets. If we fillna(0) we lie to
# the model on 98 % of the training set — that's why the vanilla baseline
# hovers at 0.56 AUC. Report parsing recovers real signal for those rows.
#
# Reports may be in English / Spanish / French / German — we include common
# terms for each. Simple sentence-level negation ("no ACL tear", "sin lesión")
# flips a match to 0.

_LABEL_KEYWORDS: dict[str, list[str]] = {
    "ACL": [r"\bACL\b", r"anterior cruciate", r"cruzado anterior",
            r"vorderes kreuzband", r"ligament croisé antérieur"],
    "MCL": [r"\bMCL\b", r"medial collateral", r"colateral medial",
            r"innenband", r"ligament collatéral médial"],
    "Medial Meniscus":  [r"medial meniscus", r"menisco medial", r"innenmeniskus",
                         r"ménisque médial", r"ménisque interne"],
    "Lateral Meniscus": [r"lateral meniscus", r"menisco lateral", r"außenmeniskus",
                         r"ménisque latéral", r"ménisque externe"],
    # OA labels: radiologists rarely write "arthritis" — much more often
    # "degenerative", "joint space narrowing", "chondromalacia", "cartilage
    # loss/thinning", "osteophytes/spurring". Per-compartment, all four langs.
    "Medial OA":  [
        r"medial (compartment )?(osteo)?arthr",
        r"medial (compartment )?(joint space narrow|jsn)",
        r"medial (compartment )?(cartilage (loss|thinning|defect)|chondr(al|omalacia|osis|opathy))",
        r"medial (compartment )?(degenerat|osteophyt|spurring|bone[- ]on[- ]bone)",
        r"medial (femorotibial|tibiofemoral).{0,30}(arthr|degenerat|narrow|chondr)",
        r"artrosis medial", r"gonartrosis medial", r"condropatía medial",
        r"mediale gonarthrose", r"innere gonarthrose", r"mediale chondropathie",
        r"arthrose (fémoro[- ]?tibiale médiale|médiale)", r"chondropathie médiale",
    ],
    "Lateral OA": [
        r"lateral (compartment )?(osteo)?arthr",
        r"lateral (compartment )?(joint space narrow|jsn)",
        r"lateral (compartment )?(cartilage (loss|thinning|defect)|chondr(al|omalacia|osis|opathy))",
        r"lateral (compartment )?(degenerat|osteophyt|spurring|bone[- ]on[- ]bone)",
        r"lateral (femorotibial|tibiofemoral).{0,30}(arthr|degenerat|narrow|chondr)",
        r"artrosis lateral", r"gonartrosis lateral", r"condropatía lateral",
        r"laterale gonarthrose", r"äußere gonarthrose", r"laterale chondropathie",
        r"arthrose (fémoro[- ]?tibiale latérale|latérale)", r"chondropathie latérale",
    ],
    "PF OA": [
        r"patellofemoral (osteo)?arthr",
        r"patellofemoral (joint space narrow|jsn)",
        r"patellofemoral (cartilage (loss|thinning|defect)|chondr(al|omalacia|osis|opathy))",
        r"patellofemoral (degenerat|osteophyt|spurring)",
        r"chondromalacia patellae?", r"patellar chondr",
        r"trochlear (cartilage|chondr)", r"retropatellar (chondr|cartilage|arthr|degenerat)",
        r"artrosis (patelo|patelofemoral|femoropatelar)",
        r"condropatía (patelo|patelofemoral|rotuliana)",
        r"retropatellare arthrose", r"femoropatellare (arthrose|chondropathie)",
        r"arthrose fémoro[- ]?patellaire", r"chondropathie rotulienne",
    ],
    "Effusion":  [r"effusion", r"joint fluid", r"derrame", r"erguss", r"épanchement"],
    "Synovitis": [r"synoviti", r"sinoviti", r"synovialiti", r"synovitis"],
    "Baker's":   [r"baker'?s? cyst", r"popliteal cyst", r"quiste de baker",
                  r"bakerzyste", r"kyste de baker", r"kyste poplité"],
    "Contusion": [r"contusion", r"bone bruise", r"bone marrow (o?edema|edema)",
                  r"contusión", r"knochenmarksödem", r"contusion osseuse"],
    "Fracture":  [r"fractur", r"fisura ósea", r"knochenbruch", r"fraktur",
                  r"break in the (cortex|bone)"],
}

_NEG_TOKENS = (
    r"\bno\b", r"\bwithout\b", r"\babsence of\b", r"\bintact\b", r"\bnegative for\b",
    r"\bsin\b", r"\bausencia de\b", r"\bkein\b", r"\bkeine\b", r"\bohne\b",
    r"\bpas de\b", r"\bsans\b", r"\baucun\b", r"\bnegatif\b",
)
_NEG_RE = re.compile("|".join(_NEG_TOKENS), re.IGNORECASE)


def _labels_from_report(text) -> dict[str, float]:
    """Return {label -> 1.0 | 0.0 | nan} from one free-text report."""
    out: dict[str, float] = {c: float("nan") for c in LABEL_COLS}
    if not isinstance(text, str) or not text.strip():
        return out
    # Split into rough sentences so negation only fires within a local window.
    sentences = re.split(r"[.\n;·•]+", text)
    for sent in sentences:
        s = sent.strip()
        if not s:
            continue
        negated = bool(_NEG_RE.search(s))
        for lbl, patterns in _LABEL_KEYWORDS.items():
            for p in patterns:
                if re.search(p, s, re.IGNORECASE):
                    val = 0.0 if negated else 1.0
                    prev = out[lbl]
                    # Positive evidence wins over negative and over NaN.
                    if math.isnan(prev) or (prev == 0.0 and val == 1.0):
                        out[lbl] = val
                    break
    return out


def _merge_expert_and_report_labels(train_csv: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """
    Build the true label matrix by combining expert labels (rows where the CSV
    already has non-NaN values) with report-derived labels for the rest.

    Report labels come from the ported multilingual parser (report_parser.extract)
    as (score, confidence) pairs. Score > 0.5 → positive; confidence is
    returned per row/label so it can be piped into sample_weight during
    training (weak/low-conf rows pull less than strong-conf ones).

    Returns (labels_df, is_expert bool Series, conf_df).
    """
    df = train_csv.set_index("StudyInstanceUID")
    labels = df[LABEL_COLS].copy()
    is_expert = labels.notna().any(axis=1)
    # Expert rows always full confidence; weak rows get 1.0 in the binary
    # parser branch (no per-row confidence) or the parser's own conf in the
    # advanced branch.
    conf = pd.DataFrame(1.0, index=labels.index, columns=LABEL_COLS)

    if USE_ADVANCED_PARSER:
        # Lazy import — the notebook builder concatenates report_parser into the
        # same cell before this file, so the name resolves at runtime.
        try:
            from report_parser import extract as _rp_extract  # local file
        except Exception:
            _rp_extract = extract  # type: ignore[name-defined]  # from concatenated cell
    else:
        _rp_extract = None  # binary regex path uses _labels_from_report

    if "Report" in df.columns:
        need = labels.index[~is_expert]
        for sid in need:
            if _rp_extract is None:
                # v10 path: binary regex parser, no per-label confidence.
                bin_labels = _labels_from_report(df.loc[sid, "Report"])
                for c in LABEL_COLS:
                    v = bin_labels.get(c, float("nan"))
                    labels.loc[sid, c] = 0 if math.isnan(v) else int(v)
                continue
            r = _rp_extract(df.loc[sid, "Report"])
            for c in LABEL_COLS:
                labels.loc[sid, c] = 1 if r.get(c, 0.0) > 0.5 else 0
                conf.loc[sid, c] = float(r.get(c + "__conf", 0.05))

    labels = labels.fillna(0).astype(int)
    return labels, is_expert, conf


# %% [dicom utilities] ------------------------------------------------------
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

# pydicom needs pylibjpeg/gdcm for JPEG-Lossless / JPEG-2000 transfer
# syntaxes. Both are preinstalled on Kaggle's Python image; we import them
# lazily and swallow errors so the script still runs on a bare environment
# (uncompressed / Implicit VR files will still decode).
for _mod in ("pylibjpeg", "gdcm"):
    try:
        __import__(_mod)
    except Exception:
        pass


def _safe_read_dcm(path: Path) -> pydicom.dataset.FileDataset | None:
    try:
        return pydicom.dcmread(str(path), force=True)
    except Exception:
        return None


def _pixels_or_none(ds) -> np.ndarray | None:
    """Return the raw calibrated float pixel array, or None on failure.

    Applies DICOM RescaleSlope + RescaleIntercept and inverts MONOCHROME1
    (where BRIGHT means LOW signal). Does NOT normalise intensity — the
    caller must window per-series, not per-slice, otherwise a slice
    through a large effusion is stretched onto the same range as a dry
    bone slice and effusion/synovitis contrast is deleted (see
    wguesdon/dinov2-at-meniscus-resolution cell 24).
    """
    try:
        arr = ds.pixel_array
    except Exception:
        return None
    if arr is None or arr.size == 0:
        return None
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[arr.shape[0] // 2]
    if arr.ndim != 2:
        return None
    try:
        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        arr = arr * slope + intercept
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            arr = float(np.nanmax(arr)) - arr
    except Exception:
        pass
    if not np.isfinite(arr).any():
        return None
    return arr


def _series_window(arrays: list[np.ndarray], lo_pct=1.0, hi_pct=99.0) -> tuple[float, float]:
    """1st..99th percentile intensity window over pooled slices of ONE series.
    Preserves relative brightness across slices (effusion still looks bright
    vs bone), unlike per-slice min/max."""
    pool = []
    for a in arrays:
        flat = a[::3, ::3].ravel()
        flat = flat[np.isfinite(flat)]
        if flat.size:
            pool.append(flat)
    if not pool:
        return 0.0, 1.0
    pool = np.concatenate(pool)
    lo, hi = np.percentile(pool, [lo_pct, hi_pct])
    if not (np.isfinite(lo) and np.isfinite(hi)) or hi <= lo:
        return float(pool.min()), float(pool.max()) if float(pool.max()) > float(pool.min()) else float(pool.min()) + 1.0
    return float(lo), float(hi)


def _apply_window(arr: np.ndarray, lo: float, hi: float) -> np.ndarray:
    """Window an intensity array to [0, 1] with given lo/hi cutoffs."""
    return np.clip((arr - lo) / max(hi - lo, 1e-6), 0.0, 1.0).astype(np.float32)


def _slice_features(img: np.ndarray) -> np.ndarray:
    """Compact per-slice descriptor: stats + histogram + edge density."""
    flat = img.ravel()
    stats = [
        float(flat.mean()),
        float(flat.std()),
        float(np.percentile(flat, 10)),
        float(np.percentile(flat, 50)),
        float(np.percentile(flat, 90)),
        float((flat > 0.5).mean()),                 # bright-tissue fraction
    ]
    hist, _ = np.histogram(flat, bins=N_HIST_BINS, range=(0.0, 1.0))
    hist = hist.astype(np.float32) / max(flat.size, 1)
    # Cheap edge-density proxy: mean absolute gradient magnitude.
    gy = np.abs(np.diff(img, axis=0)).mean() if img.shape[0] > 1 else 0.0
    gx = np.abs(np.diff(img, axis=1)).mean() if img.shape[1] > 1 else 0.0
    return np.concatenate([stats, hist, [gx, gy, img.shape[0], img.shape[1]]])


# Names come from the per-slice descriptor above; keep in sync if changed.
_PIXEL_STAT_NAMES = (
    ["mean", "std", "p10", "p50", "p90", "bright_frac"]
    + [f"hist{i}" for i in range(N_HIST_BINS)]
    + ["grad_x", "grad_y", "rows", "cols"]
)


# %% [pretrained CNN feature extractor] -------------------------------------
# ResNet18 penultimate features (512-dim) computed on GPU. These are far more
# scanner-invariant than raw pixel statistics because ImageNet training
# saw huge intensity/contrast diversity. We try to set it up at startup; if
# torch / torchvision / the weights file are missing, we transparently fall
# back to pixel statistics so the notebook still submits.
class _CNNState:
    mode: str = "pixel_stats"          # "cnn" once activated
    feature_size: int = len(_PIXEL_STAT_NAMES)
    feature_names: list[str] = list(_PIXEL_STAT_NAMES)
    model = None                        # torch.nn.Module
    device = None                       # "cuda" or "cpu"
    transform = None                    # callable(np.ndarray) -> torch.Tensor

_CNN = _CNNState()


def _torch_load_safe(path, map_location="cpu"):
    """torch.load with weights_only=False for trusted Kaggle-hosted pretrained
    weights; PyTorch 2.6+ requires this for legacy .tar pickles. Old torch
    versions don't take the kwarg — TypeError → fall through."""
    import torch
    try:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=map_location)


def _try_setup_dinov2(torch, transforms) -> bool:
    """Prefer DINOv2 (Meta self-supervised ViT — dominant medical-imaging
    backbone in 2025-26 public solutions). Requires timm + a Kaggle-hosted
    dinov2 weights .pth attached. Returns True if activated.
    """
    try:
        import timm
    except Exception:
        return False
    # Search for any DINOv2 checkpoint under /kaggle/input.
    weight_paths = list(Path("/kaggle/input").rglob("dinov2*.pth")) \
                 + list(Path("/kaggle/input").rglob("dinov2*.bin")) \
                 + list(Path("/kaggle/input").rglob("dinov2*.safetensors"))
    if not weight_paths:
        return False
    # Prefer the smallest ViT to keep runtime realistic on T4.
    weight_paths.sort(key=lambda p: (0 if "vits14" in p.name else
                                     1 if "vitb14" in p.name else
                                     2 if "vitl14" in p.name else 3))
    weight_path = weight_paths[0]
    # Map filename → timm model name and feature dim.
    name_lookup = [
        ("vits14", "vit_small_patch14_dinov2.lvd142m", 384),
        ("vitb14", "vit_base_patch14_dinov2.lvd142m", 768),
        ("vitl14", "vit_large_patch14_dinov2.lvd142m", 1024),
    ]
    tag_model_dim = next(((t, m, d) for (t, m, d) in name_lookup
                          if t in weight_path.name), None)
    if tag_model_dim is None:
        return False
    tag, model_name, feat_dim = tag_model_dim
    try:
        # ViT-S/14 was pretrained at 518×518. We want 224 for speed/memory
        # (224/14 = 16 patches exactly). Passing img_size=224 makes timm
        # rebuild the patch embed and interpolate the position embeddings
        # instead of asserting the resolution mismatch.
        net = timm.create_model(model_name, pretrained=False, num_classes=0,
                                img_size=224)
        state = _torch_load_safe(weight_path)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        # DINOv2 checkpoints from Meta sometimes have "teacher.backbone." prefix.
        state = {k.replace("teacher.backbone.", "").replace("backbone.", ""): v
                 for k, v in state.items()}
        # pos_embed shape depends on img_size — original 518 → 37×37+1 = 1370
        # tokens, our 224 → 16×16+1 = 257. Interpolate rather than discard so
        # DINOv2's pretrained positional info actually reaches the model.
        if "pos_embed" in state:
            import torch.nn.functional as F
            pretrained_pe = state["pos_embed"]                # (1, N_old, D)
            target_shape = net.pos_embed.shape                # (1, N_new, D)
            if pretrained_pe.shape != target_shape:
                cls = pretrained_pe[:, :1]                    # (1, 1, D)
                patch_pe = pretrained_pe[:, 1:]               # (1, N-1, D)
                old = int(round(patch_pe.shape[1] ** 0.5))
                new = int(round((target_shape[1] - 1) ** 0.5))
                patch_pe = (patch_pe.reshape(1, old, old, -1)
                                    .permute(0, 3, 1, 2))
                patch_pe = F.interpolate(patch_pe, size=(new, new),
                                         mode="bicubic", align_corners=False)
                patch_pe = (patch_pe.permute(0, 2, 3, 1)
                                    .reshape(1, new * new, -1))
                state["pos_embed"] = torch.cat([cls, patch_pe], dim=1)
        missing, unexpected = net.load_state_dict(state, strict=False)
        if len(unexpected) > 5:
            print(f"[cnn] DINOv2: {len(unexpected)} unexpected keys, may indicate wrong variant")
    except Exception as e:
        print(f"[cnn] DINOv2 load failed ({e}); trying ResNet18 next.")
        return False
    net.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    net = net.to(device)
    _CNN.model = net
    _CNN.device = device
    _CNN.feature_size = feat_dim
    _CNN.feature_names = [f"dv2{i:04d}" for i in range(feat_dim)]
    _CNN.mode = f"dinov2_{tag}"
    # DINOv2 uses ImageNet norm at 224 (or multiple-of-14 resolution).
    _CNN.transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    print(f"[cnn] enabled: DINOv2/{tag} @ {device}, weights={weight_path.name}, "
          f"feature_size={feat_dim}")
    return True


def _try_setup_resnet18(torch, transforms) -> bool:
    from torchvision import models
    weight_paths = list(Path("/kaggle/input").rglob("resnet18*.pth"))
    if not weight_paths:
        return False
    weight_path = weight_paths[0]
    try:
        net = models.resnet18(weights=None)
        state = _torch_load_safe(weight_path)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        net.load_state_dict(state, strict=False)
    except Exception as e:
        print(f"[cnn] ResNet18 load failed ({e}); using pixel_stats.")
        return False
    net.fc = torch.nn.Identity()
    net.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    net = net.to(device)
    _CNN.model = net
    _CNN.device = device
    _CNN.feature_size = 512
    _CNN.feature_names = [f"r18_{i:03d}" for i in range(512)]
    _CNN.mode = "resnet18"
    _CNN.transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    print(f"[cnn] enabled: ResNet18 @ {device}, weights={weight_path.name}, feature_size=512")
    return True


def _setup_cnn() -> None:
    """Prefer DINOv2 → ResNet18 → pixel_stats. Silent no-op on any failure."""
    try:
        import torch
        from torchvision import transforms
    except Exception as e:
        print(f"[cnn] torch/torchvision unavailable ({e}); using pixel_stats.")
        return
    if _try_setup_dinov2(torch, transforms):
        return
    if _try_setup_resnet18(torch, transforms):
        return
    print("[cnn] no pretrained weights found under /kaggle/input; using pixel_stats.")


def _extract_slice_features_batch(imgs_3ch: list[np.ndarray]) -> np.ndarray:
    """
    imgs_3ch: list of (H, W, 3) float32 in [0,1]. Returns (N, feature_size).
    Uses the CNN when active; pixel_stats (on the center channel) otherwise.
    """
    if not imgs_3ch:
        return np.zeros((0, _CNN.feature_size), dtype=np.float32)
    if _CNN.mode.startswith(("dinov2", "resnet18")):
        import torch
        tensors = []
        for im in imgs_3ch:
            u8 = np.clip(im * 255.0, 0, 255).astype(np.uint8)  # (H, W, 3)
            tensors.append(_CNN.transform(u8))
        batch = torch.stack(tensors, dim=0).to(_CNN.device)
        with torch.no_grad():
            feats = _CNN.model(batch)
        return feats.detach().cpu().numpy().astype(np.float32)
    # Pixel-stats fallback uses only the center (green) channel.
    return np.stack([_slice_features(im[:, :, 1]) for im in imgs_3ch],
                    axis=0).astype(np.float32)


def _series_ordered_paths(series_dir: Path) -> list[Path]:
    """Sort DICOM files by physical through-plane position (not filename).

    Filenames are SOP Instance UIDs — spearman(filename_rank, physical
    position) ≈ 0.009 measured over this corpus. Sorting by them picks
    random slices for 2.5D neighbours and 'start/middle/end' sampling.

    Correct order: project ImagePositionPatient onto the slice normal
    (cross of the two ImageOrientationPatient vectors). Falls back to
    InstanceNumber, then natural filename sort.
    """
    files = list(series_dir.glob("*.dcm"))
    if not files:
        return []
    keyed = []
    fallback_used = 0
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(str(f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient",
                                                "ImageOrientationPatient",
                                                "InstanceNumber"])
            iop = getattr(ds, "ImageOrientationPatient", None)
            ipp = getattr(ds, "ImagePositionPatient", None)
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                iop_arr = np.asarray(iop[:6], dtype=np.float64)
                ipp_arr = np.asarray(ipp[:3], dtype=np.float64)
                n = np.cross(iop_arr[:3], iop_arr[3:])
                k = float(np.dot(ipp_arr, n))
            if k is None or not np.isfinite(k):
                inst = getattr(ds, "InstanceNumber", None)
                if inst is not None:
                    k = float(inst)
                    fallback_used += 1
        except Exception:
            pass
        keyed.append((k, f))
    # If most slices lack geometry, fall back to natural filename sort
    if sum(1 for k, _ in keyed if k is None) > len(keyed) // 2:
        return sorted(files, key=lambda p: p.name)
    # Missing entries slot to +inf so they end up at the tail rather than
    # being dropped.
    return [f for _, f in sorted(keyed, key=lambda t: (t[0] if t[0] is not None else float("inf"), t[1].name))]


def _sample_slice_indices(n: int, k: int) -> list[int]:
    """Sample k slice positions spread over the CENTRAL BAND of a stack.

    Outermost knee slices are mostly soft tissue outside the joint; findings
    live in the middle. Use 15%..85% central band, matching the 0.899 recipe's
    default SLICE_BAND.
    """
    if n <= 0:
        return []
    if n <= k:
        return list(range(n))
    lo = int(0.15 * (n - 1))
    hi = int(0.85 * (n - 1))
    if hi <= lo:
        return [n // 2] * k
    return list(dict.fromkeys(int(round(x)) for x in np.linspace(lo, hi, k)))


# A small allowlist of safe DICOM header fields. We deliberately do NOT feed
# Manufacturer/model into the model — those are the documented shortcut for
# site memorisation — but we still capture them separately for grouped CV.
_SCANNER_FIELDS = ("Manufacturer", "ManufacturerModelName", "SoftwareVersions")


def _series_dicom_meta(ds) -> dict:
    """Return a dict of {feature_name: value} from one DICOM header."""
    def _num(tag, default=np.nan):
        v = getattr(ds, tag, None)
        try:
            return float(v)
        except Exception:
            return default

    return {
        "meta_repetition_time": _num("RepetitionTime"),
        "meta_echo_time": _num("EchoTime"),
        "meta_slice_thickness": _num("SliceThickness"),
        "meta_pixel_spacing_0": _num_from_seq(ds, "PixelSpacing", 0),
        "meta_pixel_spacing_1": _num_from_seq(ds, "PixelSpacing", 1),
        "meta_magnetic_field": _num("MagneticFieldStrength"),
        "meta_imaging_frequency": _num("ImagingFrequency"),
    }


def _num_from_seq(ds, tag: str, idx: int) -> float:
    v = getattr(ds, tag, None)
    try:
        return float(v[idx])
    except Exception:
        return float("nan")


def _scanner_key(ds) -> str:
    parts = []
    for f in _SCANNER_FIELDS:
        v = getattr(ds, f, None)
        parts.append(str(v) if v is not None else "?")
    return "|".join(parts)


# %% [per-study feature extraction] -----------------------------------------
def _empty_slice_vec() -> np.ndarray:
    return np.full(_CNN.feature_size, np.nan, dtype=np.float32)


def _aggregate_series_arr(arr: np.ndarray) -> np.ndarray:
    """Mean + max across slices in one series. Input (N, F). Output (2F,).
    Max captures the peak per-feature response — a nonparametric proxy for
    MIL attention pooling, which is what top solutions use for the
    "abnormality shows up on just one slice" case."""
    if arr.size == 0:
        empty = _empty_slice_vec()
        return np.concatenate([empty, empty])
    return np.concatenate([np.nanmean(arr, axis=0), np.nanmax(arr, axis=0)])


def _per_series_len() -> int:
    return 2 * _CNN.feature_size


def _features_for_study(study_dir: Path,
                        series_rows: pd.DataFrame) -> tuple[dict, str]:
    """
    Extract a feature dict for one study.

    Returns (features, scanner_key). scanner_key is used only for grouped CV,
    never as a model input.
    """
    # Aggregate per-plane so different anatomical views become distinct blocks
    # in the feature vector.
    plane_vecs: dict[str, list[np.ndarray]] = {p: [] for p in PLANES}
    meta_accum: list[dict] = []
    scanner_key = "unknown"

    # Cap the number of series considered per study (long tail exists).
    considered = series_rows.head(MAX_SERIES_PER_STUDY)
    for _, row in considered.iterrows():
        series_dir = study_dir / row["SeriesInstanceUID"]
        if not series_dir.is_dir():
            continue
        # Geometry-sorted paths — filenames are anatomically random (see
        # _series_ordered_paths docstring).
        ordered = _series_ordered_paths(series_dir)
        if not ordered:
            continue
        # Sample from the central band (findings live there; edges are soft
        # tissue). For 2.5D we also need each sampled slice's ±1 neighbours.
        sampled = _sample_slice_indices(len(ordered), SLICES_PER_SERIES)
        needed = set(sampled)
        if USE_25D_INPUT:
            for i in sampled:
                needed.add(max(0, i - 1))
                needed.add(min(len(ordered) - 1, i + 1))
        # Read raw pixels once per needed slice.
        raw_by_idx: dict[int, np.ndarray] = {}
        first_ds = None
        for i in sorted(needed):
            ds = _safe_read_dcm(ordered[i])
            if ds is None:
                continue
            if first_ds is None:
                first_ds = ds
            img = _pixels_or_none(ds)
            if img is None:
                continue
            raw_by_idx[i] = img
        if not raw_by_idx:
            continue
        if first_ds is not None and scanner_key == "unknown":
            scanner_key = _scanner_key(first_ds)
            meta_accum.append(_series_dicom_meta(first_ds))
        # ONE intensity window per series (over all decoded slices), so a
        # slice through effusion stays visibly brighter than a slice through
        # bone — crucial for Effusion / Synovitis / Baker's contrast.
        lo, hi = _series_window(list(raw_by_idx.values()))
        windowed = {i: _apply_window(a, lo, hi) for i, a in raw_by_idx.items()}
        # Build 2.5D triplets (prev / current / next) or replicated grayscale.
        slice_inputs: list[np.ndarray] = []
        for i in sampled:
            if i not in windowed:
                continue
            curr = windowed[i]
            if USE_25D_INPUT:
                prev = windowed.get(max(0, i - 1), curr)
                nxt = windowed.get(min(len(ordered) - 1, i + 1), curr)
                if prev.shape == curr.shape and nxt.shape == curr.shape:
                    slice_inputs.append(np.stack([prev, curr, nxt], axis=-1))
                    continue
            slice_inputs.append(np.stack([curr, curr, curr], axis=-1))
        # One batched call — GPU inference is only cheap when batched.
        slice_arr = _extract_slice_features_batch(slice_inputs)
        series_vec = _aggregate_series_arr(slice_arr)
        # Fluid_Sensitive / Fat_Suppression / Plane give us obvious buckets.
        plane = row.get("Anatomical_Plane", "Sagittal")
        if plane not in plane_vecs:
            plane = "Sagittal"
        plane_vecs[plane].append(series_vec)

    feats: dict[str, float] = {}
    for plane in PLANES:
        if plane_vecs[plane]:
            block = np.nanmean(np.stack(plane_vecs[plane], axis=0), axis=0)
            block_has = 1.0
        else:
            block = np.full(_per_series_len(), np.nan, dtype=np.float32)
            block_has = 0.0
        for i, v in enumerate(block):
            feats[f"{plane}_{i:02d}"] = float(v)
        feats[f"{plane}_present"] = block_has

    # Series-count features (very cheap, sometimes informative).
    feats["n_series"] = float(len(considered))
    for p in PLANES:
        feats[f"n_series_{p}"] = float((considered["Anatomical_Plane"] == p).sum())
    feats["n_fluid_sensitive"] = float(considered.get("Fluid_Sensitive", pd.Series(dtype=int)).fillna(0).sum())
    feats["n_fat_suppression"] = float(considered.get("Fat_Suppression", pd.Series(dtype=int)).fillna(0).sum())

    # DICOM header timing/geometry features — DISABLED by default because
    # v5 CV showed they leak scanner identity (see USE_SCANNER_METADATA_FEATURES
    # comment in the config block). We still collect them so scanner_key is
    # populated for the grouped-CV split; we just don't feed them to the model.
    if USE_SCANNER_METADATA_FEATURES:
        if meta_accum:
            meta_df = pd.DataFrame(meta_accum)
            for c in meta_df.columns:
                feats[c] = float(meta_df[c].mean(skipna=True))
        else:
            for c in ("meta_repetition_time", "meta_echo_time", "meta_slice_thickness",
                      "meta_pixel_spacing_0", "meta_pixel_spacing_1",
                      "meta_magnetic_field", "meta_imaging_frequency"):
                feats[c] = float("nan")

    return feats, scanner_key


def _build_feature_frame(studies_csv: pd.DataFrame,
                         series_csv: pd.DataFrame,
                         root: Path,
                         subsample: int | None = None) -> tuple[pd.DataFrame, pd.Series]:
    if subsample:
        studies_csv = studies_csv.sample(
            n=min(subsample, len(studies_csv)),
            random_state=RANDOM_STATE,
        ).reset_index(drop=True)
    series_by_study = series_csv.groupby("StudyInstanceUID")

    rows, scanners = [], []
    t0 = time.time()
    for i, sid in enumerate(studies_csv["StudyInstanceUID"].tolist()):
        study_dir = root / sid
        try:
            srows = series_by_study.get_group(sid)
        except KeyError:
            srows = pd.DataFrame(columns=series_csv.columns)
        feats, sk = _features_for_study(study_dir, srows)
        feats["StudyInstanceUID"] = sid
        rows.append(feats)
        scanners.append(sk)
        if (i + 1) % 25 == 0 or i + 1 == len(studies_csv):
            elapsed = time.time() - t0
            print(f"  [{i+1}/{len(studies_csv)}] studies processed "
                  f"({elapsed:.1f}s, {(i+1)/max(elapsed, 1e-6):.2f} studies/s)")
    df = pd.DataFrame(rows).set_index("StudyInstanceUID")
    return df, pd.Series(scanners, index=df.index, name="scanner_key")


# %% [training / validation] ------------------------------------------------
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold


def _fit_predict_per_label(X_train: pd.DataFrame,
                           Y_train: pd.DataFrame,
                           X_test: pd.DataFrame,
                           sample_weight: np.ndarray | None = None) -> np.ndarray:
    """Train one HistGradientBoosting per label; return P(test, 12).

    sample_weight (if given) up-weights expert-labeled rows over weak
    report-derived rows. Same weights are applied to every per-label model.
    """
    preds = np.full((len(X_test), len(LABEL_COLS)), 0.5, dtype=np.float32)
    for j, col in enumerate(LABEL_COLS):
        y = Y_train[col].astype(int).values
        if y.sum() == 0 or y.sum() == len(y):
            preds[:, j] = float(y.mean())
            continue
        clf = HistGradientBoostingClassifier(
            max_iter=300, max_depth=6, learning_rate=0.05,
            l2_regularization=1.0, random_state=RANDOM_STATE,
        )
        clf.fit(X_train.values, y, sample_weight=sample_weight)
        preds[:, j] = clf.predict_proba(X_test.values)[:, 1]
    return preds


def _macro_auc(y_true: pd.DataFrame, y_pred: np.ndarray) -> tuple[float, dict]:
    per_label: dict[str, float] = {}
    for j, col in enumerate(LABEL_COLS):
        yt = y_true[col].astype(int).values
        if yt.sum() == 0 or yt.sum() == len(yt):
            per_label[col] = float("nan")
            continue
        per_label[col] = float(roc_auc_score(yt, y_pred[:, j]))
    macro = float(np.nanmean(list(per_label.values())))
    return macro, per_label


def _cv_score(X: pd.DataFrame, Y: pd.DataFrame,
              splitter, groups=None,
              sample_weight: np.ndarray | None = None) -> tuple[float, list[float]]:
    fold_aucs: list[float] = []
    oof = np.zeros((len(X), len(LABEL_COLS)), dtype=np.float32)
    split_args = (X, Y.sum(axis=1) > 0) if groups is None else (X, Y.sum(axis=1) > 0, groups)
    for fold, (tr, va) in enumerate(splitter.split(*split_args)):
        sw = sample_weight[tr] if sample_weight is not None else None
        preds = _fit_predict_per_label(X.iloc[tr], Y.iloc[tr], X.iloc[va],
                                       sample_weight=sw)
        oof[va] = preds
        m, _ = _macro_auc(Y.iloc[va], preds)
        fold_aucs.append(m)
        print(f"    fold {fold+1}: macro AUC = {m:.4f}")
    overall, per_label = _macro_auc(Y, oof)
    print(f"    overall OOF macro AUC = {overall:.4f}")
    for col, v in per_label.items():
        print(f"      {col:>17s}: {v:.4f}")
    return overall, fold_aucs


# %% [main] -----------------------------------------------------------------
def main() -> None:
    print("== RSNA Knee Baseline ==")
    print(f"DATA_DIR = {DATA_DIR}")
    print(f"OUT_DIR  = {OUT_DIR}")
    _setup_cnn()  # ResNet18 GPU features when weights are attached; else pixel_stats
    print(f"Slice feature mode = {_CNN.mode} (feature_size={_CNN.feature_size})")

    required = ["train.csv", "train_series.csv",
                "test.csv", "test_series.csv", "sample_submission.csv"]
    missing = [f for f in required if not (DATA_DIR / f).is_file()]
    if missing:
        print(f"ERROR: could not find {missing} under {DATA_DIR}.")
        print("Contents of /kaggle/input (top level):")
        try:
            for p in Path("/kaggle/input").iterdir():
                print(f"  {p}")
        except Exception:
            pass
        raise FileNotFoundError(
            f"Missing {missing} in {DATA_DIR}. Set RSNA_DATA_DIR env var "
            "to the directory that contains train.csv."
        )

    train_csv = pd.read_csv(DATA_DIR / "train.csv")
    train_series = pd.read_csv(DATA_DIR / "train_series.csv")
    test_csv = pd.read_csv(DATA_DIR / "test.csv")
    test_series = pd.read_csv(DATA_DIR / "test_series.csv")
    sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

    # --- 1. Brief EDA --------------------------------------------------
    print(f"\ntrain.csv: {train_csv.shape}, cols={list(train_csv.columns)[:6]}...")
    print("Per-label positive rate:")
    for c in LABEL_COLS:
        if c in train_csv.columns:
            print(f"  {c:>17s}: {train_csv[c].mean():.3f} "
                  f"({int(train_csv[c].sum())} positives)")
    print(f"\ntrain_series.csv: {train_series.shape}")
    if "Anatomical_Plane" in train_series.columns:
        print("  planes:", train_series["Anatomical_Plane"].value_counts().to_dict())
    series_per_study = train_series.groupby("StudyInstanceUID").size()
    print(f"  series per study: median={series_per_study.median():.0f}, "
          f"p95={series_per_study.quantile(0.95):.0f}, "
          f"max={series_per_study.max():.0f}")
    if "PatientSex" in train_csv.columns:
        miss = train_csv["PatientSex"].isna().sum() + (train_csv["PatientSex"] == "").sum()
        print(f"  PatientSex missing/blank: {miss}/{len(train_csv)}")
    else:
        print("  PatientSex column absent (schema notes warn this can happen)")

    # --- 1b. Build labels: expert rows + weak labels from Report ------
    # This is the highest-leverage step. Without it, ~4349 of 4407 rows
    # have NaN labels that fillna(0) turns into fabricated negatives,
    # which caps validation AUC around 0.56 no matter what features we use.
    all_labels, is_expert, label_conf = _merge_expert_and_report_labels(train_csv)
    n_expert = int(is_expert.sum())
    n_weak = int((~is_expert & (all_labels.sum(axis=1) > 0)).sum())
    n_empty = int((~is_expert & (all_labels.sum(axis=1) == 0)).sum())
    print(f"\nLabel coverage after merging Report-derived weak labels:")
    print(f"  expert-labeled rows      : {n_expert}")
    print(f"  report-derived (any pos) : {n_weak}")
    print(f"  no-signal rows (all zero): {n_empty}  (kept as negatives)")
    print("  merged per-label positive rate:")
    for c in LABEL_COLS:
        print(f"    {c:>17s}: {all_labels[c].mean():.3f} "
              f"({int(all_labels[c].sum())} positives)")

    # --- 2/E2E. End-to-end DINOv2 + SlotHead path ----------------------
    if USE_END2END:
        try:
            from end2end import train_and_predict as _e2e_run  # local file
        except Exception:
            # Concatenated-cell fallback (notebook builder inlines the module).
            _e2e_run = train_and_predict  # type: ignore[name-defined]

        # Studies to use for training — everything with any label signal
        # (i.e. drop the ~634 all-zero rows to avoid drowning the loss in noise).
        train_ids_all = [sid for sid in all_labels.index
                         if bool(is_expert.loc[sid]) or all_labels.loc[sid].sum() > 0]
        print(f"\n[e2e] training set: {len(train_ids_all)} studies "
              f"({int(is_expert.sum())} expert + rest report-derived)")

        # Simple holdout for macro-AUC readout (5-fold too expensive here).
        rng = np.random.default_rng(RANDOM_STATE)
        perm = rng.permutation(len(train_ids_all))
        n_hold = int(E2E_HOLDOUT * len(train_ids_all))
        hold_ids = [train_ids_all[i] for i in perm[:n_hold]]
        fit_ids = [train_ids_all[i] for i in perm[n_hold:]]
        test_ids = list(test_csv["StudyInstanceUID"])

        # First: train on `fit_ids`, predict holdout, report macro AUC.
        print(f"[e2e] fit on {len(fit_ids)}, holdout {len(hold_ids)}")
        hold_preds = _e2e_run(
            fit_ids, hold_ids,
            series_df=train_series, root_train=DATA_DIR / "train_series",
            root_test=DATA_DIR / "train_series",     # holdout is drawn from train
            labels_df=all_labels, conf_df=label_conf, is_expert=is_expert,
            pixel_loader=_pixels_or_none,
            ordered_paths_fn=_series_ordered_paths,
            sampler_fn=_sample_slice_indices,
            series_window_fn=_series_window,
            apply_window_fn=_apply_window,
        )
        hold_y = all_labels.reindex(hold_ids).values
        per_label = []
        for j, col in enumerate(LABEL_COLS):
            y = hold_y[:, j].astype(int)
            if y.sum() == 0 or y.sum() == len(y):
                continue
            per_label.append(roc_auc_score(y, hold_preds[:, j]))
        holdout_auc = float(np.mean(per_label)) if per_label else float("nan")
        print(f"[e2e] holdout macro AUC = {holdout_auc:.4f} "
              f"(over {len(per_label)}/12 labels with both classes present)")

        # Then: train on ALL data (fit + hold) and predict real test.
        print(f"[e2e] final fit on all {len(train_ids_all)} training studies")
        test_preds_arr = _e2e_run(
            train_ids_all, test_ids,
            series_df=pd.concat([train_series, test_series], ignore_index=True),
            root_train=DATA_DIR / "train_series",
            root_test=DATA_DIR / "test_series",
            labels_df=all_labels, conf_df=label_conf, is_expert=is_expert,
            pixel_loader=_pixels_or_none,
            ordered_paths_fn=_series_ordered_paths,
            sampler_fn=_sample_slice_indices,
            series_window_fn=_series_window,
            apply_window_fn=_apply_window,
        )
        pred_df = pd.DataFrame(test_preds_arr, columns=LABEL_COLS, index=test_ids)
        random_auc = holdout_auc
        grouped_auc = float("nan")   # end-to-end skips scanner-grouped CV for now
        _write_submission(pred_df, sample_sub, all_labels)
        _print_e2e_summary(random_auc)
        return

    # --- 2. Feature extraction ----------------------------------------
    print("\nExtracting TRAIN features...")
    X_train, scanner_train = _build_feature_frame(
        train_csv, train_series,
        root=DATA_DIR / "train_series",
        subsample=TRAIN_SUBSAMPLE,
    )
    Y_train = all_labels.reindex(X_train.index).fillna(0).astype(int)
    # Sample weight combines two signals: a 3× base multiplier for expert rows
    # (ground truth vs weak labels) AND the parser's per-row confidence
    # (weak rows where the parser was highly confident count more than rows
    # where it was hedging). Sample weight is scalar per row, so we take the
    # mean confidence across the 12 labels for that row.
    expert_flag = is_expert.reindex(X_train.index).fillna(False).values
    if USE_ADVANCED_PARSER:
        row_conf = label_conf.reindex(X_train.index).fillna(0.05).mean(axis=1).values
        base = np.where(expert_flag, 3.0, 1.0)
        sample_weight = (base * row_conf).astype(np.float32)
    else:
        # v10 path: flat 3× expert, 1× weak — no per-row confidence.
        sample_weight = np.where(expert_flag, 3.0, 1.0).astype(np.float32)

    print("\nExtracting TEST features...")
    X_test, _ = _build_feature_frame(
        test_csv, test_series,
        root=DATA_DIR / "test_series",
        subsample=None,
    )

    # Align columns; HistGradientBoosting handles NaN natively.
    all_cols = sorted(set(X_train.columns) | set(X_test.columns))
    X_train = X_train.reindex(columns=all_cols)
    X_test = X_test.reindex(columns=all_cols)

    # --- 3. Validation: random-stratified + scanner-grouped -----------
    print("\nCV #1: random StratifiedKFold on (has-any-label)...")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    random_auc, _ = _cv_score(X_train, Y_train, skf, groups=None,
                              sample_weight=sample_weight)

    print("\nCV #2: GroupKFold by scanner (Manufacturer|Model|Software)...")
    n_groups = scanner_train.nunique()
    if n_groups >= 2:
        n_splits = int(min(N_FOLDS, n_groups))
        gkf = GroupKFold(n_splits=n_splits)
        grouped_auc, _ = _cv_score(X_train, Y_train, gkf,
                                   groups=scanner_train.values,
                                   sample_weight=sample_weight)
    else:
        grouped_auc = float("nan")
        print("  Only one scanner group present — grouped CV skipped.")

    if not math.isnan(grouped_auc) and (random_auc - grouped_auc) > 0.05:
        print("\n  WARNING: random-fold AUC >> grouped-fold AUC — the model is "
              "leaning on scanner/site identity. Prune metadata features or add"
              " site-invariant training before trusting the leaderboard.")

    # --- 4. Final fit on all training data, predict on test -----------
    print("\nFinal fit on all training studies, predicting test...")
    test_preds = _fit_predict_per_label(X_train, Y_train, X_test,
                                        sample_weight=sample_weight)
    pred_df = pd.DataFrame(test_preds, columns=LABEL_COLS, index=X_test.index)

    # --- 5. Build submission matching sample_submission.csv exactly ---
    sub = sample_sub.copy()
    id_col = "StudyInstanceUID"
    pred_df = pred_df.reindex(sub[id_col].values)
    for c in LABEL_COLS:
        vals = pred_df[c].values
        # Any study we couldn't score falls back to the label prior — better
        # than 0.5 and keeps AUC well-defined.
        prior = float(Y_train[c].mean()) if c in Y_train else 0.5
        vals = np.where(np.isnan(vals), prior, vals)
        sub[c] = np.clip(vals, 0.0, 1.0).astype(np.float32)

    sub_path = OUT_DIR / "submission.csv"
    sub.to_csv(sub_path, index=False)
    print(f"\nWrote {sub_path} — {sub.shape[0]} rows, {sub.shape[1]} cols")

    # --- 6. Plain-language summary ------------------------------------
    print("\n" + "=" * 66)
    print("SUMMARY")
    print("=" * 66)
    print(f"Random 5-fold macro ROC AUC : {random_auc:.4f}")
    print(f"Scanner-grouped macro AUC   : {grouped_auc:.4f}")
    print("\nModel: per-study features = per-plane pixel statistics "
          "(mean/std/percentiles/histogram/edge density) + safe DICOM-header "
          "numbers → one HistGradientBoostingClassifier per label. Labels "
          "are expert-annotated where available, else derived from the "
          "free-text Report via multilingual keyword + sentence-level "
          "negation; expert rows get 3x sample-weight in training.")
    print("\nNext steps to lift the score further, in rough order of value:")
    print("  1. Replace pixel stats with a pretrained CNN feature extractor"
          " (ResNet18/EfficientNet from an attached torchvision-weights"
          " Kaggle dataset) — expect +0.05 to +0.10 macro AUC.")
    print("  2. Improve report parsing: LLM-based label extraction (offline"
          " model attached as a dataset) is much stronger than regex on"
          " noisy multilingual text.")
    print("  3. Train separate per-plane sub-models (Sagittal / Coronal /"
          " Axial), blend with per-label optimal weights, and add TTA"
          " (flip + neighbouring-slice averaging).")


def _write_submission(pred_df, sample_sub, all_labels):
    """Reindex predictions to sample_submission.csv order, clip [0,1], write."""
    sub = sample_sub.copy()
    id_col = "StudyInstanceUID"
    pred_df = pred_df.reindex(sub[id_col].values)
    for c in LABEL_COLS:
        vals = pred_df[c].values
        prior = float(all_labels[c].mean()) if c in all_labels else 0.5
        vals = np.where(np.isnan(vals), prior, vals)
        sub[c] = np.clip(vals, 0.0, 1.0).astype(np.float32)
    sub_path = OUT_DIR / "submission.csv"
    sub.to_csv(sub_path, index=False)
    print(f"\nWrote {sub_path} — {sub.shape[0]} rows, {sub.shape[1]} cols")


def _print_e2e_summary(holdout_auc):
    print("\n" + "=" * 66)
    print("SUMMARY (end-to-end DINOv2 + SlotHead)")
    print("=" * 66)
    print(f"Holdout macro ROC AUC : {holdout_auc:.4f}")
    print("\nModel: DINOv2 ViT-S/14 with last 2 blocks unfrozen; 3 slots per study "
          "(Sagittal/Coronal/Axial, longest series per plane), 3 slices per slot "
          "(central-band, geometry-ordered, per-series windowed). SlotHead: per-label "
          "attention over slot embeddings with anatomy prior. Labels are expert + "
          "multilingual-parser-derived; weak rows down-weighted by parser confidence.")
    print("\nNext levers (from the 0.899 recipe, not yet in v15):")
    print("  1. 6-slot scheme (plane × fat-suppression × fluid-weighting) instead of 3.")
    print("  2. Laterality normalisation (flip right knees to left) via IPP+IOP.")
    print("  3. Unfreeze 6 backbone blocks + 10 epochs instead of 2 + 3 epochs.")
    print("  4. 336×336 input resolution (currently 224×224); ~2x compute but +AUC.")


def _write_fallback_submission() -> None:
    """Guarantee submission.csv exists even if the pipeline crashed early."""
    out = OUT_DIR / "submission.csv"
    if out.is_file():
        return
    src = DATA_DIR / "sample_submission.csv"
    if src.is_file():
        pd.read_csv(src).to_csv(out, index=False)
        print(f"Fallback submission written from {src} -> {out}")


if __name__ == "__main__":
    try:
        main()
    except Exception:
        import traceback
        traceback.print_exc()
        _write_fallback_submission()
        raise
